In [0]:
from pyspark.sql.functions import current_timestamp, col

# Define parameters via widgets
dbutils.widgets.text("catalog", "dbr_dev", "1. Catalog Name")
dbutils.widgets.text("schema", "valeriimatviiv_bronze", "2. Target Schema")
dbutils.widgets.text("volume", "market_radar_landing", "3. Landing Volume")
dbutils.widgets.text("max_files_per_trigger", "1000", "4. Max Files Per Trigger")

# Retrieve widget values
catalog = dbutils.widgets.get("catalog")
schema = dbutils.widgets.get("schema")
volume = dbutils.widgets.get("volume")
max_files_per_trigger = dbutils.widgets.get("max_files_per_trigger")

base_volume_path = f"/Volumes/{catalog}/{schema}/{volume}"
landing_news_path = f"{base_volume_path}/landing/finnhub_news"
checkpoint_news_path = f"{base_volume_path}/_state/checkpoints/finnhub_news"
schema_news_path = f"{base_volume_path}/_state/schemas/finnhub_news"

target_table = f"{catalog}.{schema}.finnhub_news_bronze"

# Reset checkpoint/schema state once to process raw landing files cleanly
dbutils.fs.rm(checkpoint_news_path, recurse=True)
dbutils.fs.rm(schema_news_path, recurse=True)
spark.sql(f"DROP TABLE IF EXISTS {target_table}")

# Configure Auto Loader Stream (Lab 3 Pattern)
df_news_stream = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "json")
    .option("cloudFiles.schemaLocation", schema_news_path)
    .option("cloudFiles.schemaEvolutionMode", "addNewColumns")
    .option("cloudFiles.maxFilesPerTrigger", max_files_per_trigger)
    .load(landing_news_path)
    .withColumn("_source_file", col("_metadata.file_path"))
    .withColumn("_ingest_timestamp", current_timestamp())
)

# Write Micro-Batch Stream to Delta Bronze Table
query = (
    df_news_stream.writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", checkpoint_news_path)
    .trigger(availableNow=True)
    .toTable(target_table)
)

query.awaitTermination()
print(f"Streaming execution completed for Delta table: {target_table}")

In [0]:
# catalog = dbutils.widgets.get("catalog")
# schema = dbutils.widgets.get("schema")
# target_table = f"{catalog}.{schema}.finnhub_news_bronze"

# df_bronze_news = spark.read.table(target_table)

# print(f"--- Bronze News Table Record Count: {df_bronze_news.count()} ---")
# print("--- Schema Breakdown ---")
# df_bronze_news.printSchema()

# print("--- Preview Ingested Data ---")
# display(df_bronze_news.limit(10))

In [0]:
# catalog = dbutils.widgets.get("catalog")
# schema = dbutils.widgets.get("schema")
# target_table = f"{catalog}.{schema}.finnhub_news_bronze"

# df = spark.read.table(target_table)

# print("--- Schema Evolution Breakdown by Phase ---")
# display(
#     df.groupBy("_schema_phase")
#     .count()
# )

# print("--- Sample showing evolved field 'index_tracker' ---")
# display(df.select("id", "_schema_phase", "index_tracker", "related", "headline").limit(10))